# C3 - Day 1: Hirsch Dataset Exploration
**AnchorAI | Component 3: Cognitive Checkpointing**

Goal: understand exactly what the Hirsch et al. *resumption costs* data contains **before** any modelling. Every column meaning below comes from the dataset's `Codebook.pdf`.

> Run on **CPU runtime** in Google Colab. Upload the three `raw-data_Exp*.csv` files to `MyDrive/C3/data/raw/hirsch/` first.

## Step 0 - Mount Google Drive and set paths

In [ ]:
import os, csv, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/C3'
    print('Running in Colab - Drive mounted.')
except ImportError:
    BASE = os.environ.get('C3_BASE', r'C:\Users\user\OneDrive\Documents\AnchorAI\AnchorAI\components\c3_checkpointing')
    print('Running locally.')

DATA_DIR   = os.path.join(BASE, 'data', 'raw', 'hirsch')
PROC_DIR   = os.path.join(BASE, 'data', 'processed')
OUTPUT_DIR = os.path.join(BASE, 'outputs')
os.makedirs(PROC_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Raw data folder:', DATA_DIR)
print('Files:', sorted(os.listdir(DATA_DIR)))

## Step 1 - Load the three experiments
The three files have two quirks:
1. Every row is wrapped in extra double-quotes, so we parse them with a small helper.
2. Column names differ between experiments (`bBlock` vs `block`, `Trialtype` vs `trialtype`). We rename them so all three tables look the same.

We keep only the 9 columns we need. The stimulus/colour/response columns describe the lab screen, not the interruption, so we ignore them.

In [ ]:
def load_hirsch_csv(path):
    """Read a Hirsch CSV whose every row is wrapped in outer double-quotes."""
    with open(path, 'r', encoding='utf-8') as f:
        lines = f.read().splitlines()

    def parse(line):
        line = line.strip()
        if line.startswith('"') and line.endswith('"'):
            line = line[1:-1].replace('""', '"')
        return next(csv.reader([line]))

    header = parse(lines[0])
    rows = [parse(l) for l in lines[1:] if l.strip()]
    return pd.DataFrame(rows, columns=header)


KEEP = ['pp', 'block', 'blocktype', 'trial', 'trialtype', 'task', 'sequence', 'RT', 'error']
frames = []
for exp, fname in [('Exp1', 'raw-data_Exp1.csv'), ('Exp2', 'raw-data_Exp2.csv'), ('Exp3', 'raw-data_Exp3.csv')]:
    d = load_hirsch_csv(os.path.join(DATA_DIR, fname))
    d = d.rename(columns={'bBlock': 'block', 'Trialtype': 'trialtype'})   # make the names identical
    d = d[KEEP].copy()
    d['experiment'] = exp
    d['row_order'] = np.arange(len(d))
    frames.append(d)

raw = pd.concat(frames, ignore_index=True)
for c in KEEP:
    raw[c] = pd.to_numeric(raw[c], errors='coerce')
raw['pp_id'] = raw['experiment'] + '_' + raw['pp'].astype(int).astype(str)   # participant id, unique across experiments

print(raw.groupby('experiment').agg(rows=('RT', 'size'), participants=('pp_id', 'nunique')))
print('Total rows:', len(raw), '| total participants:', raw['pp_id'].nunique())

## Step 2 - What does each column mean? (from Codebook.pdf)

| Column | Meaning |
|--------|---------|
| `pp` | Participant ID |
| `block` | Block number |
| `blocktype` | **1 = practice, 2 = experimental** |
| `trial` | Trial number within a block |
| `trialtype` | **1 = trial without interruption, 2 = trial with interruption** |
| `task` | **0 = the interruption task**, 1-3 = the three subtasks of the main procedure (Exp3 has a 4th subtask) |
| `sequence` | 1 = the last subtask repeats the previous subtask, 2 = the last subtask switches |
| `RT` | Reaction time of this response, in milliseconds |
| `error` | 0 = correct response, 1 = wrong response |

**Important:** one row is *one response* (one step). One *trial* is a short procedure of 3 or 4 steps. So a row's `RT` is the time for that single step, not a whole task.

In [ ]:
raw.head(8)

## Step 3 - Participants and sample sizes
`GroupKFold`-style splitting later needs a participant ID that is unique. We build `pp_id = experiment + pp`. We also check that the same participant number is not reused across experiments.

In [ ]:
ids_per_exp = {e: set(g['pp']) for e, g in raw.groupby('experiment')}
print('Participants per experiment:', {e: len(s) for e, s in ids_per_exp.items()})
print('Overlap of raw pp numbers between experiments:',
      len(ids_per_exp['Exp1'] & ids_per_exp['Exp2']),
      len(ids_per_exp['Exp1'] & ids_per_exp['Exp3']),
      len(ids_per_exp['Exp2'] & ids_per_exp['Exp3']))
print()
print('Rows per participant (min / median / max):',
      raw.groupby('pp_id').size().agg(['min', 'median', 'max']).tolist())

## Step 4 - Practice versus experimental blocks
`blocktype = 1` is practice: participants are still learning the procedure, so their reaction times are not representative. We keep **experimental blocks only** (`blocktype = 2`).

In [ ]:
print(raw['blocktype'].map({1: 'practice', 2: 'experimental'}).value_counts())

## Step 5 - What does one trial look like? (the key discovery)
We group rows into trials and look at the order of the `task` values.

* **Uninterrupted trial**: subtask 1 -> subtask 2 -> subtask 3 (-> subtask 4 in Exp3).
* **Interrupted trial**: subtask 1 -> subtask 2 -> **interruption task (0)** -> subtask 3 (-> subtask 4).

So the **resumption step is subtask 3**: the first thing the participant does after coming back from the interruption. That is the reaction time we want to understand.

In [ ]:
g = raw[raw['blocktype'] == 2].sort_values('row_order').groupby(['experiment', 'pp_id', 'block', 'trial'])['task']
patterns = g.apply(lambda s: ' -> '.join(str(int(t)) for t in s)).reset_index(name='task_order')
print(patterns.groupby(['experiment', 'task_order']).size().rename('trials').reset_index().to_string(index=False))

ex = raw[(raw['experiment'] == 'Exp1') & (raw['pp_id'] == raw['pp_id'].iloc[0]) & (raw['blocktype'] == 2)]
print('\nExample: one interrupted and one uninterrupted trial of the first participant (Exp1)')
for tt in (2, 1):
    one = ex[ex['trialtype'] == tt]
    first = one[(one['block'] == one['block'].iloc[0]) & (one['trial'] == one['trial'].iloc[0])]
    print(f'\ntrialtype = {tt}')
    print(first[['block', 'trial', 'trialtype', 'task', 'sequence', 'RT', 'error']].to_string(index=False))

## Step 6 - Data-quality check: does `trialtype` match what really happened?
The codebook says `trialtype = 2` means "with interruption". We verify this against the data: an interrupted trial must contain a task-0 response.

In [ ]:
chk = raw[raw['blocktype'] == 2].groupby(['experiment', 'pp_id', 'block', 'trial']).agg(
    label_says_interrupted=('trialtype', lambda s: s.iloc[0] == 2),
    has_task0=('task', lambda s: (s == 0).any())).reset_index()
chk['mismatch'] = chk['label_says_interrupted'] != chk['has_task0']
print(chk.groupby('experiment')['mismatch'].agg(['sum', 'count']).rename(columns={'sum': 'mismatching_trials', 'count': 'trials'}))
print('\nParticipants affected:', chk.loc[chk['mismatch'], 'pp_id'].unique().tolist())

**Finding:** the labels agree with the data in Exp1 and Exp2, but a few Exp3 trials disagree. Because the *actual structure* is what matters, from Day 2 on we define "interrupted" as **"the trial contains a task-0 response"** and do not rely on `trialtype`.

## Step 7 - Raw reaction-time range
Reaction time is right-skewed and has extreme values. Before using it we must decide how to trim it (done in Day 2).

In [ ]:
rt = raw['RT']
print(f'RT range: {rt.min():.0f} - {rt.max():.0f} ms | median {rt.median():.0f} ms | mean {rt.mean():.0f} ms')
print(f'Responses faster than 200 ms: {(rt < 200).sum():,} | slower than 10 s: {(rt > 10000).sum():,}')

plt.figure(figsize=(8, 4))
plt.hist(rt[rt <= 8000], bins=80, color='steelblue', edgecolor='white')
plt.title('Reaction time of all responses (clipped at 8000 ms for display)')
plt.xlabel('RT (ms)'); plt.ylabel('Count')
plt.tight_layout(); plt.show()

## Step 8 - Does an interruption really slow the resumption step?
This checks the whole premise. We compare the reaction time of **subtask 3** in interrupted trials (right after the interruption) with subtask 3 in uninterrupted trials. Only correct responses are used. If interruptions really cost time, the interrupted bars should be higher.

We first rebuild one row per trial (`rt0`, `rt1`, ... = reaction time of task 0, 1, ...; `err0`, ... = errors).

In [ ]:
KEY = ['experiment', 'pp_id', 'block', 'trial']
exp_rows = raw[raw['blocktype'] == 2]                       # experimental blocks only

rt_wide  = exp_rows.pivot(index=KEY, columns='task', values='RT').add_prefix('rt')       # rt0, rt1, rt2, rt3 (rt4)
err_wide = exp_rows.pivot(index=KEY, columns='task', values='error').add_prefix('err')   # err0, err1, ...
info     = exp_rows.groupby(KEY).agg(sequence=('sequence', 'first'), trialtype=('trialtype', 'first'))

trials = pd.concat([rt_wide, err_wide, info], axis=1).reset_index()
trials['interrupted'] = trials['rt0'].notna()                # True if the interruption task (task 0) really happened
print('Trials (experimental blocks):', len(trials))
print(trials['interrupted'].value_counts().rename({True: 'interrupted', False: 'uninterrupted'}))

In [ ]:
ok = trials[trials['err3'] == 0]
med = ok.groupby(['experiment', 'interrupted'])['rt3'].median().unstack()
med.columns = ['uninterrupted', 'interrupted']
med['resumption_cost_ms'] = med['interrupted'] - med['uninterrupted']
print(med.round(0))

per_pp = ok.groupby(['pp_id', 'interrupted'])['rt3'].median().unstack().dropna()
print(f'\n{(per_pp[True] > per_pp[False]).mean() * 100:.1f}% of participants are slower right after an interruption.')

med[['uninterrupted', 'interrupted']].plot(kind='bar', figsize=(6, 4), color=['#8fb8de', '#d9534f'])
plt.title('Median RT of subtask 3 (correct responses)')
plt.ylabel('RT (ms)'); plt.xticks(rotation=0)
plt.tight_layout(); plt.show()

## Summary - what we learned today (numbers are printed above)
1. The data holds **148 participants in 3 experiments**. One row is one response (one step), one trial is 3-4 steps.
2. Practice blocks are excluded; interrupted trials are recognised by the presence of the interruption task (task 0).
3. The **resumption step is subtask 3**. Its reaction time is slower after an interruption than without one, which is the effect C3 wants to predict.
4. A few Exp3 `trialtype` labels disagree with the real trial structure, so we use the structure.
5. Raw RT contains extreme values, so trimming is needed.

**Next: Day 2 - build the resumption dataset: target (resumption cost), features, participant-wise split.**